## 1 · Übersicht

Dieses Notebook bestimmt das Asset-Universum für die gesamte Analyse: welche Coins über
den betrachteten Zeitraum als "relevant" gelten und mit welchem Binance-Symbol sie später
für die OHLCV-Beschaffung (`02_ohlcv.ipynb`) verwendet werden.

Es baut das Universum **pro vollständigem Lauf einmal komplett neu** (nicht Tag für Tag).
Historische Filter werden aus gecachten Rohantworten rekonstruiert statt in einer
Live-Tag-für-Tag-Schleife berechnet.

### Ablauf (Schritte, wie im Code kommentiert)

**Schritt (a): heutiger Kandidatenpool.** Die tagesabhängigen Filter unten müssen wissen,
*welche* Coins überhaupt geprüft werden. CoinGecko bietet auf keinem Tarif einen Endpoint,
der "Top 100 nach Market Cap an einem vergangenen Datum" direkt liefert — die einzige
Möglichkeit ist, für jeden Coin dessen eigene Zeitreihe zu holen und erst danach zu ranken.
Deshalb wird der Kandidatenpool aus den **heutigen** Top `TOP_N_CANDIDATES` Coins nach
Market Cap gebildet (`/coins/markets`). Ein Coin, der vor langer Zeit Top-100 war, aber
seither aus diesem Pool gefallen ist, ist unerreichbar. Das ist eine strukturelle Einschränkung
von CoinGecko, die diese Pipeline nicht vollständig beheben kann. Ein grösserer Pool
könnte diesen Survivorship-Bias reduzieren, beseitigt ihn aber nicht. Hier ist zu bemerken, dass trotzdem das Universum über die Zeit kleiner geworden ist. Das deutet darauf hin, dass der Survivorship-Bias nicht sehr gross sein kann.

**Schritt (b): historische Zeitreihe pro Kandidat.** Pro Kandidat liefert
`/coins/{id}/market_chart/range` die komplette Tages-Zeitreihe (Preis, Market Cap,
Volumen) für den ganzen Zeitraum in einem einzigen Call. Antworten werden vor jeder
Filterung auf Platte gecacht, damit erneutes Ausführen der Filterlogik beim Entwickeln
keine zusätzlichen API-Credits kostet.

**Schritt (c): tägliche Filter 1+2, über den ganzen Zeitraum.**
- **Filter 1**: Rang nach `market_cap_usd` innerhalb des Kandidatenpools, pro Tag; Top `TOP_N_DAILY` behalten.
- **Filter 2**: nur Zeilen mit `global_volume_24h_usd > MIN_VOLUME_24H_USD` behalten.

Das ersetzt die frühere tägliche `/coins/markets?date=...`-Schleife, die sich als
fehlerhaft herausstellte: dieser Endpoint hat gar keinen `date`-Parameter und lieferte
bei jeder Iteration stillschweigend den *aktuellen* Stand (verifiziert — ein gespeicherter
BTC-Preis stand 19+ Tage am Stück still).

**Schritt (d): Union aller je-eligiblen Coins.** Jeder Coin, der an mindestens einem Tag
im Fenster Filter 1+2 erfüllt hat, wird Kandidat für Filter 3.

**Schritt (e): Filter 3, nur aktueller Stand.** Anders als Filter 1+2 wird die
Exchange-Abdeckung nur gegen den **heutigen** Stand geprüft, einmalig, für die (d)-Union
— nicht pro historischem Tag erneut. "Major Exchanges" sind die Top `TOP_EXCHANGES_N` aus
dem CoinMarketCap-Spot-Exchange-Ranking, gemappt auf CoinGecko-Exchange-IDs. Ein Asset
besteht, wenn es auf mindestens `MIN_TOP_EXCHANGE_COUNT` davon gehandelt wird.

**Schritt (f)+(g) — finale Tabellen + Binance-Symbol-Auflösung.** Dieselbe gecachte
Ticker-Antwort, die für die Exchange-Zählung verwendet wird, enthält auch den exakten
Binance-Ticker-String für den Coin (`/coins/{id}/tickers?exchange_ids=binance`),
gespeichert als `binance_symbol` — das ersetzt das Raten von `SYMBOL + "USDT"`, was echte
Matching-Lücken gegen Binances tatsächliche Listings erzeugt hat.

### Resultierende Tabellen-Semantik

**`asset`** — fixer Einmal-Output: eine Zeile pro Asset, das Filter 1+2 an mindestens
einem Tag im Fenster erfüllt hat UND aktuell Filter 3 erfüllt. Ein späterer Re-Run (z.B.
um das Fenster um ein paar Wochen zu verlängern) baut die ganze Tabelle neu, aber
`asset_id` bleibt für bereits bekannte Coins erhalten — nur wirklich neue Coins bekommen
eine neue ID (`load_existing_asset_ids()` / `build_final_assets_df`). Ein voller Re-Run
leitet auch den Kandidatenpool erneut aus dem *heutigen* Top-N ab — ein Coin, der
inzwischen herausgefallen ist, verschwindet dann komplett aus der Union, selbst für
historische Tage, an denen er korrekt eligible war (derselbe Survivorship-Bias-Tradeoff
wie in Schritt (a), nur bei jedem Re-Run erneut sichtbar).

**`asset_daily`** — tagesvariables Panel: eine Zeile pro (Datum, `asset_id`), an dem das
Asset Filter 1+2 an genau diesem Tag erfüllt hat, beschränkt auf Assets, die es in die
finale `asset`-Tabelle geschafft haben. `01_baskets.ipynb`'s Entered/Left/Switched-Logik
hängt von dieser Tag-zu-Tag-Variation ab, deshalb bleibt sie erhalten, obwohl die
OHLCV-Universe (in `02_ohlcv.ipynb`) für den ganzen Zeitraum fix ist.

### Referenzen

- CoinGecko API-Doku: [Coins Markets](https://docs.coingecko.com/reference/coins-markets) · [Market Chart Range](https://docs.coingecko.com/reference/coins-id-market-chart-range) · [Coin Tickers](https://docs.coingecko.com/reference/coins-id-tickers) · [Exchanges List](https://docs.coingecko.com/reference/exchanges-list) · [Authentication (Demo vs. Pro)](https://docs.coingecko.com/reference/authentication) · [Preise/Rate-Limits](https://www.coingecko.com/en/api/pricing)
- Gescrapte Seite für die Exchange-Rangliste: [CoinMarketCap Rankings — Exchanges](https://coinmarketcap.com/rankings/exchanges/)

### UML-Diagramm

Quelle: `docs/uml/assets_UML.drawio` (mit draw.io/diagrams.net öffnen, PNG-Export für die
Anzeige unten aktualisieren).

![UML](../../docs/uml/assets_UML.png)

## 2 · Imports

Nur Standardbibliothek + `pandas`/`requests` — keine `pycoingecko`/`ccxt`-Wrapper, alle
API-Calls sind bewusst hand-gerollt (`coingecko_get()`, Abschnitt 5), damit der
Plan-Switch (Demo/Basic) und das Caching an einer Stelle sitzen.

In [ ]:
# Imports
from pathlib import Path
import sys
import io
import time
import json
import configparser
from datetime import datetime, timezone

import pandas as pd
import requests
import matplotlib.pyplot as plt

## 3 · Konfiguration

Alles Stellschrauben-Wertige an einem Ort: geteilte Pfade (`RAW_DIR`, `FETCHED_DIR`,
`ASSETS_PATH`, `ASSET_DAILY_PATH`) und die `VS_CURRENCY`/`MIN_VOLUME_24H_USD`-Schwelle
kommen aus `notebooks/config.py` (gemeinsam mit `01_baskets.ipynb`), der `CG_PLAN`-Schalter
(steuert Base-URL, Auth-Header und Sleep-Zeit gemeinsam, siehe [Authentication-Doku](https://docs.coingecko.com/reference/authentication)),
die restlichen zwei Filter-Schwellen, das historische Fenster und die Cache-Unterverzeichnisse
sind notebook-spezifisch und bleiben hier lokal. Nichts davon ist in den Helfer-Funktionen
weiter unten hartkodiert.

In [ ]:
# Configuration

# ── Shared paths & thresholds — from config.py ──────────────────
sys.path.insert(0, str(Path("../..").resolve()))
from Src.config import (
    RAW_DIR, FETCHED_DIR,
    ASSETS_PATH, ASSET_DAILY_PATH,
    VS_CURRENCY, MIN_VOLUME_24H_USD,
)

# ── CoinGecko plan switch ──────────────────────────────────────
# "demo"  -> free Demo key, api.coingecko.com, 365-day historical cap, ~30 calls/min
# "basic" -> paid Basic plan, pro-api.coingecko.com, 2-year historical depth, ~300 calls/min
CG_PLAN = "basic"

CG_PLAN_CONFIG = {
    "demo": {
        "base_url": "https://api.coingecko.com/api/v3",
        "header_name": "x-cg-demo-api-key",
        "secret_section": "COINGECKO",
        "secret_option": "DEMO_API_KEY",
        "sleep_seconds": 2.1,
        "max_history_days": 365,
    },
    "basic": {
        "base_url": "https://pro-api.coingecko.com/api/v3",
        "header_name": "x-cg-pro-api-key",
        "secret_section": "COINGECKO",
        "secret_option": "PRO_API_KEY",
        "sleep_seconds": 0.21,
        "max_history_days": None,
    },
}

# ── Universe parameters ─────────────────────────────────────────
TOP_N_CANDIDATES = 100   # today's candidate pool seeded from /coins/markets (survivorship-bias buffer)
TOP_N_DAILY = 100        # daily top-N-by-market-cap filter (filter 1), applied within the candidate pool
TOP_EXCHANGES_N = 20     # how many CMC-ranked exchanges define "major exchanges"

MIN_TOP_EXCHANGE_COUNT = 10       # filter 3 threshold (checked once, current state only)
# MIN_VOLUME_24H_USD (filter 2 threshold) comes from config.py — shared with 01_baskets.ipynb

# USD stablecoins excluded from the candidate pool entirely (before filters 1+2).
# Rationale: this universe feeds a pairs-trading/cointegration strategy. A coin pinned to
# $1 doesn't carry tradeable price signal, would "cointegrate" with other stablecoins
# trivially (peg-arbitrage noise, not a real spread), and pollutes distance-based
# clustering (TDA) by forming a trivial cluster of near-identical flat series. They also
# occupy top-market-cap slots that would otherwise go to genuinely volatile assets in the
# TOP_N_DAILY ranking. Not exhaustive by design -- add ids here as new ones show up.
STABLECOIN_DENYLIST = {
    "tether",            # USDT
    "usd-coin",          # USDC
    "dai",               # DAI
    "usds",              # USDS
    "usd1-wlfi",         # USD1
    "ethena-usde",       # USDe
    "true-usd",          # TUSD
    "usdd",              # USDD
    "frax",              # FRAX
    "paxos-standard",    # USDP
    "first-digital-usd", # FDUSD
    "gemini-dollar",     # GUSD
    "binance-usd",       # BUSD
}

# Optional dev knob: cap the candidate pool for a cheap first pass. None = use TOP_N_CANDIDATES.
DEV_CANDIDATE_LIMIT: int | None = None

# ── Historical window ────────────────────────────────────────────
# Real run: full 2 years, matching the CoinGecko Basic plan's max historical depth.
START_DATE = "2024-07-23"
END_DATE = "2026-07-22"  # inclusive

# ── Cache paths (notebook-specific, not shared via config.py) ───
CACHE_DIR = RAW_DIR / "cache" / "coingecko"

# Named cache subdirectories (single source of truth, used by every fetch helper below
# instead of repeating the literal folder name at each call site).
CACHE_CANDIDATES_DIR = CACHE_DIR / "candidates"
CACHE_MARKET_CHART_DIR = CACHE_DIR / "market_chart"
CACHE_TICKERS_DIR = CACHE_DIR / "tickers_topexchanges"

# ── Cache behavior ───────────────────────────────────────────────
FORCE_REFETCH_CACHE = False  # True = ignore existing cache files and hit the network again
BATCH_SIZE = 25              # coins per batch for market_chart + tickers fetching, with interim checkpoints

REQUEST_TIMEOUT_SECONDS = 30
USER_AGENT = "TDA-Data-Aquisition"

config = configparser.ConfigParser()
config.read("../../secret.cfg")

_plan_cfg = CG_PLAN_CONFIG[CG_PLAN]
COINGECKO_API_KEY = config.get(_plan_cfg["secret_section"], _plan_cfg["secret_option"])

if not COINGECKO_API_KEY:
    raise ValueError(f"CoinGecko API key not found in secret.cfg for plan '{CG_PLAN}'")

print(f"CG_PLAN = {CG_PLAN} | base_url = {_plan_cfg['base_url']} | max_history_days = {_plan_cfg['max_history_days']}")

## 4 · Tabellen-Schemas

Die verbindliche Spaltenliste für beide Output-Tabellen. Jede Tabelle, die später gebaut
wird, läuft durch `enforce_columns()`/`ensure_asset_columns()` (Abschnitt 6) — Abweichungen
(fehlende oder unerwartete Spalten) brechen den Lauf sofort, statt sich stillschweigend
in `assets.parquet`/`asset_daily.parquet` einzuschleichen.

In [ ]:
# Table schemas
ASSET_COLUMNS = [
    "asset_id",
    "coingecko_id",
    "symbol",
    "name",
    "circulating_supply",
    "total_supply",
    "max_supply",
    "top_exchange_count",
    "binance_symbol",
]

ASSET_DAILY_COLUMNS = [
    "date",
    "asset_id",
    "price_usd",
    "market_cap_usd",
    "global_volume_24h_usd",
]

## 5 · CoinGecko Request-Helper & Verbindungstest

`coingecko_get()` ist der einzige Ort, an dem HTTP-Requests an CoinGecko gestellt werden.
Base-URL, Auth-Header und Sleep-Zeit kommen ausschliesslich aus `CG_PLAN_CONFIG[CG_PLAN]`
(Abschnitt 3) — ein Wechsel von Demo auf Basic ändert also nie Code, nur die eine
Konfigurationszeile. Bei HTTP 429 (Rate-Limit) wird mit steigender Wartezeit bis zu
`max_retries`-mal erneut versucht.

Referenz: [CoinGecko Authentication](https://docs.coingecko.com/reference/authentication) ·
[Rate-Limits & Pläne](https://www.coingecko.com/en/api/pricing)

In [ ]:
# Request helpers
def coingecko_get(endpoint: str, params: dict | None = None, max_retries: int = 5) -> dict | list:
    plan_cfg = CG_PLAN_CONFIG[CG_PLAN]
    url = f"{plan_cfg['base_url']}{endpoint}"
    headers = {
        "User-Agent": USER_AGENT,
        plan_cfg["header_name"]: COINGECKO_API_KEY,
    }
    params = dict(params or {})
    last_error = None

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        if response.status_code == 429:
            wait_seconds = min(90, 5 * (attempt + 1))
            print(
                f"429 for {endpoint} | attempt {attempt + 1}/{max_retries} | sleeping {wait_seconds}s"
            )
            time.sleep(wait_seconds)
            last_error = RuntimeError(f"429 rate limit for {url}")
            continue

        response.raise_for_status()
        time.sleep(plan_cfg["sleep_seconds"])
        return response.json()

    raise last_error or RuntimeError(f"Request failed for {url}")

In [ ]:
# API test
print(coingecko_get("/ping"))

## 6 · Validierung & Cache-Helfer

Generische Bausteine, die von allen späteren Fetch-Funktionen wiederverwendet werden:
Spalten-Validierung, Datumsbereichs-Prüfung (inkl. Check gegen das Plan-Limit aus
Abschnitt 3) und der Disk-Cache für Rohantworten (`read_json_cache`/`write_json_cache`).
`print_batch_progress()` ist die gemeinsame Fortschritts-Ausgabe für die batchweisen
Fetch-Schleifen in den Abschnitten 7 und 8.

In [ ]:
# Validation and cache helpers
def ensure_output_dirs() -> None:
    FETCHED_DIR.mkdir(parents=True, exist_ok=True)
    ASSET_DAILY_PATH.parent.mkdir(parents=True, exist_ok=True)
    for cache_subdir in [CACHE_CANDIDATES_DIR, CACHE_MARKET_CHART_DIR, CACHE_TICKERS_DIR]:
        cache_subdir.mkdir(parents=True, exist_ok=True)


def enforce_columns(df: pd.DataFrame, expected_columns: list[str]) -> pd.DataFrame:
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if extra:
        raise ValueError(f"Unexpected columns: {extra}")

    return df[expected_columns].copy()


def ensure_asset_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "top_exchange_count" not in df.columns:
        df["top_exchange_count"] = pd.NA
    if "binance_symbol" not in df.columns:
        df["binance_symbol"] = pd.NA

    return enforce_columns(df, ASSET_COLUMNS)


def validate_date_range(start_date: str, end_date: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    start_dt = pd.Timestamp(start_date)
    end_dt = pd.Timestamp(end_date)

    if start_dt > end_dt:
        raise ValueError(f"START_DATE must be on or before END_DATE. Got {start_date} > {end_date}.")

    today_utc = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
    max_history_days = CG_PLAN_CONFIG[CG_PLAN]["max_history_days"]

    if max_history_days is not None:
        earliest_allowed = today_utc - pd.Timedelta(days=max_history_days)
        if start_dt < earliest_allowed:
            raise ValueError(
                f"START_DATE {start_date} is older than the '{CG_PLAN}' plan's "
                f"{max_history_days}-day historical limit (earliest allowed: {earliest_allowed.date()})."
            )

    return start_dt, end_dt


def make_date_list(start_date: str, end_date: str) -> list[str]:
    start_dt, end_dt = validate_date_range(start_date, end_date)
    return [d.strftime("%Y-%m-%d") for d in pd.date_range(start=start_dt, end=end_dt, freq="D")]


# ── Generic disk cache for raw API responses ──────────────────
# Rationale: filter/matching logic gets developed and re-run many times; caching the
# raw response once means those re-runs cost zero additional API credits.
def read_json_cache(cache_path: Path) -> dict | list | None:
    if FORCE_REFETCH_CACHE or not cache_path.exists():
        return None
    with open(cache_path, "r") as f:
        return json.load(f)


def write_json_cache(cache_path: Path, data: dict | list) -> None:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump(data, f)


# ── Shared progress line for the batched fetch loops (sections 7 + 8) ──
def print_batch_progress(label: str, done: int, total: int, fetched: int, cache_hits: int, errors_so_far: int) -> None:
    print(f"  {label} batch {done}/{total} | fetched={fetched} | cache_hit={cache_hits} | errors_so_far={errors_so_far}")

## 7 · Kandidatenpool & historische Rekonstruktion (Filter 1+2)

Vier Funktionen, die zusammen Schritt (a)–(c) aus Abschnitt 1 umsetzen:

1. `fetch_top_n_candidates()` — heutige Top-N nach Market Cap ([`/coins/markets`](https://docs.coingecko.com/reference/coins-markets)), paginiert, gecacht pro Tag.
2. `fetch_market_chart_range()` / `fetch_market_chart_for_candidates()` — pro Kandidat ein Call an [`/coins/{id}/market_chart/range`](https://docs.coingecko.com/reference/coins-id-market-chart-range) für die komplette Zeitreihe, batchweise mit Zwischenständen.
3. `build_daily_panel_from_market_charts()` — Rohantworten zu einem langen `(date, coingecko_id, price_usd, market_cap_usd, global_volume_24h_usd)`-Panel zusammenführen.
4. `apply_daily_filters()` — Filter 1 (Rang) + Filter 2 (Volumen), vektorisiert pro Tag, kein weiterer API-Call nötig.

In der Ausführungszelle (Abschnitt 11, Schritt a) wird der Kandidatenpool direkt nach dem
Fetch um `STABLECOIN_DENYLIST` bereinigt (Abschnitt 3) — Stablecoins sollen für eine
Pairs-Trading-Strategie erst gar nicht als Kandidaten in die Ranking-/Volumen-Filter
einfliessen, siehe Begründung dort.

In [ ]:
# Candidate pool + historical market-chart helpers
def fetch_top_n_candidates(top_n: int) -> pd.DataFrame:
    """Today's top-N coins by market cap. Seeds which coins get historical time series fetched at all."""
    run_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    cache_path = CACHE_CANDIDATES_DIR / f"top{top_n}_{run_date}.json"

    cached = read_json_cache(cache_path)
    if cached is not None:
        print(f"[cache] candidate pool <- {cache_path.name}")
        data = cached
    else:
        data = []
        per_page = min(top_n, 250)  # 250 is CoinGecko's max page size
        pages_needed = -(-top_n // per_page)  # ceil division
        for page in range(1, pages_needed + 1):
            page_data = coingecko_get(
                "/coins/markets",
                params={
                    "vs_currency": VS_CURRENCY,
                    "order": "market_cap_desc",
                    "per_page": per_page,
                    "page": page,
                    "sparkline": "false",
                },
            )
            if not page_data:
                break
            data.extend(page_data)
            if len(page_data) < per_page:
                break  # short page = no more results, stop before an empty final call
        write_json_cache(cache_path, data)
        print(f"[fetch] candidate pool -> {cache_path.name} ({len(data)} rows)")

    df = pd.DataFrame(data[:top_n])

    required_cols = ["id", "symbol", "name", "circulating_supply", "total_supply", "max_supply"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"CoinGecko response missing required fields: {missing_cols}")

    df = df.rename(columns={"id": "coingecko_id"})
    df["symbol"] = df["symbol"].astype(str).str.upper()

    return df[["coingecko_id", "symbol", "name", "circulating_supply", "total_supply", "max_supply"]].copy()


def fetch_market_chart_range(coingecko_id: str, from_ts: int, to_ts: int) -> tuple[dict, bool]:
    """Full daily price/market_cap/volume time series for one coin in a single call. Returns (data, was_cache_hit)."""
    cache_path = CACHE_MARKET_CHART_DIR / f"{coingecko_id}__{from_ts}_{to_ts}.json"

    cached = read_json_cache(cache_path)
    if cached is not None:
        return cached, True

    data = coingecko_get(
        f"/coins/{coingecko_id}/market_chart/range",
        params={"vs_currency": VS_CURRENCY, "from": from_ts, "to": to_ts},
    )
    write_json_cache(cache_path, data)
    return data, False


def fetch_market_chart_for_candidates(
    candidate_ids: list[str],
    from_ts: int,
    to_ts: int,
    batch_size: int = BATCH_SIZE,
) -> tuple[dict[str, dict], list[dict]]:
    charts: dict[str, dict] = {}
    errors: list[dict] = []
    n = len(candidate_ids)

    for batch_start in range(0, n, batch_size):
        batch = candidate_ids[batch_start : batch_start + batch_size]
        fetched = 0
        cache_hits = 0

        for coingecko_id in batch:
            try:
                data, was_cache_hit = fetch_market_chart_range(coingecko_id, from_ts, to_ts)
                charts[coingecko_id] = data
                if was_cache_hit:
                    cache_hits += 1
                else:
                    fetched += 1
            except Exception as e:
                errors.append({"coingecko_id": coingecko_id, "error": repr(e)})

        done = min(batch_start + batch_size, n)
        print_batch_progress("market_chart", done, n, fetched, cache_hits, len(errors))

    print(f"market_chart fetch complete: {len(charts)}/{n} coins succeeded, {len(errors)} errors")
    return charts, errors


def build_daily_panel_from_market_charts(charts: dict[str, dict]) -> pd.DataFrame:
    """Long (date, coingecko_id, price_usd, market_cap_usd, global_volume_24h_usd) panel from cached raw responses."""
    rows = []

    for coingecko_id, data in charts.items():
        prices = dict(data.get("prices", []))
        market_caps = dict(data.get("market_caps", []))
        volumes = dict(data.get("total_volumes", []))

        all_ts = set(prices) | set(market_caps) | set(volumes)
        for ts_ms in all_ts:
            date = pd.to_datetime(ts_ms, unit="ms", utc=True).normalize().tz_localize(None)
            rows.append(
                {
                    "date": date,
                    "coingecko_id": coingecko_id,
                    "price_usd": prices.get(ts_ms),
                    "market_cap_usd": market_caps.get(ts_ms),
                    "global_volume_24h_usd": volumes.get(ts_ms),
                }
            )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # market_chart/range can return more than one point per calendar day for recent dates
    # (hourly granularity) -> collapse to one row per (date, coin), keeping the latest observation.
    df = df.sort_values("date").drop_duplicates(subset=["date", "coingecko_id"], keep="last")
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df.reset_index(drop=True)


def apply_daily_filters(
    panel_df: pd.DataFrame,
    top_n_daily: int,
    min_volume_24h_usd: float,
) -> pd.DataFrame:
    """Filter 1 (top-N by market cap, ranked within the candidate pool) + filter 2 (24h volume), applied per day."""
    df = panel_df.copy()
    df["market_cap_usd"] = pd.to_numeric(df["market_cap_usd"], errors="coerce")
    df["global_volume_24h_usd"] = pd.to_numeric(df["global_volume_24h_usd"], errors="coerce")
    df["price_usd"] = pd.to_numeric(df["price_usd"], errors="coerce")

    df = df.dropna(subset=["market_cap_usd"])
    # method="first": deterministic tie-break by row order, ties are only ever a
    # theoretical edge case here (two coins with byte-identical market cap on the same day)
    df["market_cap_rank"] = df.groupby("date")["market_cap_usd"].rank(method="first", ascending=False)

    rank_mask = df["market_cap_rank"] <= top_n_daily
    volume_mask = df["global_volume_24h_usd"] > min_volume_24h_usd

    eligible = df.loc[rank_mask & volume_mask].drop(columns=["market_cap_rank"])
    return eligible.reset_index(drop=True)

## 8 · Top-Exchange-Abdeckung & Binance-Symbol (Filter 3)

Setzt Schritt (e)–(g) aus Abschnitt 1 um:

- `fetch_top_exchange_names_from_cmc()` scrapt die [CoinMarketCap-Exchange-Rangliste](https://coinmarketcap.com/rankings/exchanges/) (gecacht als HTML).
- `fetch_coingecko_exchange_id_map()` holt CoinGecko's [`/exchanges/list`](https://docs.coingecko.com/reference/exchanges-list), um die CMC-Namen auf CoinGecko-Exchange-IDs zu mappen.
- `fetch_tickers_for_asset_cached()` fragt einmal pro Asset [`/coins/{id}/tickers`](https://docs.coingecko.com/reference/coins-id-tickers) ab (paginiert, gecacht) — **eine** Rohantwort, die danach für zwei Zwecke wiederverwendet wird:
  - `count_top_exchanges()` → Filter-3-Schwelle
  - `resolve_binance_symbol()` → exakter Binance-Ticker-String (ersetzt das frühere `SYMBOL + "USDT"`-Raten)

In [ ]:
# Top-exchange + Binance-symbol helpers (filter 3, current state only)
def fetch_top_exchange_names_from_cmc(top_n: int) -> pd.DataFrame:
    cache_path = CACHE_DIR / "cmc_exchange_rankings.html"

    cached_html = None
    if not FORCE_REFETCH_CACHE and cache_path.exists():
        cached_html = cache_path.read_text()

    if cached_html is None:
        resp = requests.get(
            "https://coinmarketcap.com/rankings/exchanges/",
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        resp.raise_for_status()
        cached_html = resp.text
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(cached_html)
        print(f"[fetch] CMC exchange rankings -> {cache_path.name}")
    else:
        print(f"[cache] CMC exchange rankings <- {cache_path.name}")

    tables = pd.read_html(io.StringIO(cached_html))
    if not tables:
        raise ValueError("Could not read exchange table from CoinMarketCap rankings page.")

    cmc_df = tables[0].copy()

    name_col = None
    for candidate in cmc_df.columns:
        candidate_str = str(candidate).strip().lower()
        if "exchange" in candidate_str or candidate_str == "name":
            name_col = candidate
            break

    if name_col is None:
        raise ValueError(
            f"Could not find exchange-name column in CoinMarketCap table. Columns: {list(cmc_df.columns)}"
        )

    out = cmc_df[[name_col]].copy()
    out.columns = ["exchange_name"]
    out["exchange_name"] = out["exchange_name"].astype(str).str.strip()
    out = out.loc[out["exchange_name"] != ""].head(top_n).reset_index(drop=True)

    return out


def fetch_coingecko_exchange_id_map() -> pd.DataFrame:
    cache_path = CACHE_DIR / "exchanges_list.json"
    cached = read_json_cache(cache_path)

    if cached is not None:
        print(f"[cache] exchange list <- {cache_path.name}")
        exchange_list = cached
    else:
        exchange_list = coingecko_get("/exchanges/list")
        write_json_cache(cache_path, exchange_list)
        print(f"[fetch] exchange list -> {cache_path.name}")

    exchange_map_df = pd.DataFrame(exchange_list)

    required_cols = ["id", "name"]
    missing_cols = [c for c in required_cols if c not in exchange_map_df.columns]
    if missing_cols:
        raise ValueError(f"CoinGecko exchange list missing columns: {missing_cols}")

    exchange_map_df = exchange_map_df[required_cols].copy()
    exchange_map_df["name_norm"] = (
        exchange_map_df["name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return exchange_map_df


def map_cmc_exchanges_to_coingecko_ids(
    cmc_exchanges_df: pd.DataFrame,
    coingecko_exchange_map_df: pd.DataFrame,
) -> pd.DataFrame:
    cmc_df = cmc_exchanges_df.copy()
    cg_df = coingecko_exchange_map_df.copy()

    cmc_df["name_norm"] = cmc_df["exchange_name"].astype(str).str.strip().str.lower()

    mapped_df = cmc_df.merge(
        cg_df[["id", "name", "name_norm"]],
        on="name_norm",
        how="left",
    )

    mapped_df = mapped_df.rename(
        columns={
            "id": "coingecko_exchange_id",
            "name": "coingecko_exchange_name",
        }
    )

    return mapped_df


def fetch_tickers_for_asset_cached(coingecko_id: str, exchange_ids: list[str]) -> tuple[list[dict], bool]:
    """Paginated /coins/{id}/tickers for the given exchange_ids, cached as one merged list.

    Reused for both filter 3 (top-exchange count) and Binance-symbol resolution, so each
    eligible asset only needs a single ticker fetch, not two.
    """
    cache_path = CACHE_TICKERS_DIR / f"{coingecko_id}.json"
    cached = read_json_cache(cache_path)
    if cached is not None:
        return cached, True

    page = 1
    all_tickers: list[dict] = []
    while True:
        tickers_data = coingecko_get(
            f"/coins/{coingecko_id}/tickers",
            params={"exchange_ids": ",".join(exchange_ids), "page": page},
        )
        tickers = tickers_data.get("tickers", [])
        if not tickers:
            break
        all_tickers.extend(tickers)
        if len(tickers) < 100:
            break  # short page = last page, CoinGecko paginates at 100/page
        page += 1

    write_json_cache(cache_path, all_tickers)
    return all_tickers, False


def count_top_exchanges(tickers: list[dict], exchange_ids: list[str]) -> int:
    matched_exchange_ids = set()
    for ticker in tickers:
        market = ticker.get("market") or {}
        exchange_id = market.get("identifier")
        if exchange_id in exchange_ids:
            matched_exchange_ids.add(exchange_id)
    return len(matched_exchange_ids)


def resolve_binance_symbol(tickers: list[dict]) -> str | None:
    """Exact Binance ticker string for an asset, derived from CoinGecko's own ticker data
    instead of guessing `SYMBOL + "USDT"` (which produced real matching gaps previously)."""
    binance_tickers = [t for t in tickers if (t.get("market") or {}).get("identifier") == "binance"]
    if not binance_tickers:
        return None

    # Prefer the most liquid/standard quote currency available, in this order.
    preferred_targets = ["USDT", "USDC", "BUSD", "FDUSD"]
    for target in preferred_targets:
        for t in binance_tickers:
            if str(t.get("target", "")).upper() == target:
                return f"{str(t.get('base', '')).upper()}{target}"

    # No preferred quote currency found -> fall back to whatever pair exists.
    t = binance_tickers[0]
    return f"{str(t.get('base', '')).upper()}{str(t.get('target', '')).upper()}"

In [ ]:
# Filter 3 (current top-exchange coverage) + asset_id attachment
def fetch_filter3_data_for_union(
    union_ids: list[str],
    exchange_ids: list[str],
    batch_size: int = BATCH_SIZE,
) -> tuple[pd.DataFrame, list[dict]]:
    """Applied ONLY to the (smaller) union of ever-eligible coins, not the full candidate pool."""
    rows = []
    errors: list[dict] = []
    n = len(union_ids)

    for batch_start in range(0, n, batch_size):
        batch = union_ids[batch_start : batch_start + batch_size]
        fetched = 0
        cache_hits = 0

        for coingecko_id in batch:
            try:
                tickers, was_cache_hit = fetch_tickers_for_asset_cached(coingecko_id, exchange_ids)
                rows.append(
                    {
                        "coingecko_id": coingecko_id,
                        "top_exchange_count": count_top_exchanges(tickers, exchange_ids),
                        "binance_symbol": resolve_binance_symbol(tickers),
                    }
                )
                if was_cache_hit:
                    cache_hits += 1
                else:
                    fetched += 1
            except Exception as e:
                errors.append({"coingecko_id": coingecko_id, "error": repr(e)})

        done = min(batch_start + batch_size, n)
        print_batch_progress("filter-3", done, n, fetched, cache_hits, len(errors))

    result_df = pd.DataFrame(rows, columns=["coingecko_id", "top_exchange_count", "binance_symbol"])
    print(f"filter-3 fetch complete: {len(result_df)}/{n} coins succeeded, {len(errors)} errors")
    return result_df, errors


def attach_asset_ids(eligible_daily_df: pd.DataFrame, asset_df: pd.DataFrame) -> pd.DataFrame:
    df = eligible_daily_df.copy()
    asset_lookup_df = asset_df[["asset_id", "coingecko_id"]].copy()

    df = df.merge(asset_lookup_df, on="coingecko_id", how="inner")  # drops coins that failed filter 3

    df["asset_id"] = df["asset_id"].astype(int)
    return df

## 9 · Finale Tabellen & Validierungs-Helfer

`build_final_assets_df()` wendet Filter 3 an und vergibt `asset_id`s — bestehende IDs aus
einem vorherigen Lauf werden über `load_existing_asset_ids()` wiederverwendet, nur
wirklich neue Coins bekommen eine neue ID. `build_asset_daily_df_final()` beschränkt das
historische Panel auf die finalen Assets. `validate_final_tables()` prüft danach
Eindeutigkeit, Vollständigkeit und Konsistenz zwischen beiden Tabellen, bevor irgendetwas
gespeichert wird.

In [ ]:
# Final table builders (one-shot rebuild each run, but asset_id stays stable across runs)
def load_existing_asset_ids() -> pd.DataFrame:
    """coingecko_id -> asset_id mapping from the previous run's assets.parquet, if any.
    Re-running this notebook later (e.g. to pick up 2 more weeks of history) rebuilds
    everything from scratch, but downstream joins on asset_id should still point at the
    same coin — so existing ids are preserved and only genuinely new coins get new ids."""
    if not ASSETS_PATH.exists():
        return pd.DataFrame(columns=["coingecko_id", "asset_id"])
    return pd.read_parquet(ASSETS_PATH, columns=["coingecko_id", "asset_id"])


def build_final_assets_df(
    union_static_df: pd.DataFrame,
    filter3_df: pd.DataFrame,
    min_top_exchange_count: int,
    existing_ids_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """union_static_df: one row per coin that passed filters 1+2 on any day, with static metadata.
    filter3_df: current top_exchange_count + binance_symbol per union coin.
    Applies filter 3. Reuses asset_id from existing_ids_df for coins already known;
    only assigns fresh ids to coins that are new since the last run."""
    merged = union_static_df.merge(filter3_df, on="coingecko_id", how="left")
    merged["top_exchange_count"] = pd.to_numeric(merged["top_exchange_count"], errors="coerce")

    final_df = merged.loc[merged["top_exchange_count"] >= min_top_exchange_count].copy()
    final_df = final_df.sort_values("coingecko_id").reset_index(drop=True)

    if existing_ids_df is not None and not existing_ids_df.empty:
        final_df = final_df.merge(existing_ids_df[["coingecko_id", "asset_id"]], on="coingecko_id", how="left")
        next_asset_id = int(existing_ids_df["asset_id"].max()) + 1
        missing_mask = final_df["asset_id"].isna()
        missing_count = int(missing_mask.sum())
        if missing_count > 0:
            final_df.loc[missing_mask, "asset_id"] = range(next_asset_id, next_asset_id + missing_count)
        final_df["asset_id"] = final_df["asset_id"].astype(int)
    else:
        final_df.insert(0, "asset_id", range(1, len(final_df) + 1))

    final_df = final_df.sort_values("asset_id").reset_index(drop=True)
    return ensure_asset_columns(final_df)


def build_asset_daily_df_final(eligible_daily_df: pd.DataFrame, asset_df: pd.DataFrame) -> pd.DataFrame:
    """Restrict the historical daily panel to the final (post-filter-3) asset set and attach asset_id.
    Stays day-varying: an asset only has a row on days it actually passed filters 1+2."""
    df = attach_asset_ids(eligible_daily_df, asset_df)
    out = df[["date", "asset_id", "price_usd", "market_cap_usd", "global_volume_24h_usd"]].copy()
    out = out.sort_values(["date", "asset_id"]).reset_index(drop=True)
    return enforce_columns(out, ASSET_DAILY_COLUMNS)

In [ ]:
# Validation helper
def validate_final_tables(asset_df: pd.DataFrame, asset_daily_df: pd.DataFrame) -> None:
    asset_df = ensure_asset_columns(asset_df)
    asset_daily_df = enforce_columns(asset_daily_df, ASSET_DAILY_COLUMNS)

    if asset_df["asset_id"].duplicated().any():
        raise ValueError("Duplicate asset_id found in asset_df.")

    if asset_df["coingecko_id"].duplicated().any():
        raise ValueError("Duplicate coingecko_id found in asset_df.")

    if asset_df["symbol"].isna().any() or (asset_df["symbol"].astype(str).str.strip() == "").any():
        raise ValueError("Empty symbol found in asset_df.")

    if asset_df["name"].isna().any() or (asset_df["name"].astype(str).str.strip() == "").any():
        raise ValueError("Empty name found in asset_df.")

    if asset_daily_df.duplicated(subset=["date", "asset_id"]).any():
        raise ValueError("Duplicate (date, asset_id) found in asset_daily_df.")

    for col in ["price_usd", "market_cap_usd", "global_volume_24h_usd"]:
        if (pd.to_numeric(asset_daily_df[col], errors="coerce").dropna() < 0).any():
            raise ValueError(f"Negative values found in {col}.")

    if not asset_daily_df.empty:
        daily_asset_ids = set(asset_daily_df["asset_id"].astype(int).unique())
        master_asset_ids = set(asset_df["asset_id"].astype(int).unique())

        if not daily_asset_ids.issubset(master_asset_ids):
            raise ValueError("asset_daily_df contains asset_id values not present in asset_df.")

## 10 · Ausführung: Exchange-Universe & Zeitraum

Ab hier wird tatsächlich ausgeführt (vorher nur Definitionen). Zuerst die einmalige,
aktuelle Exchange-Universe (Filter 3, siehe Abschnitt 8) und der validierte Datumsbereich
inkl. der Unix-Epoch-Grenzen (`from_ts`/`to_ts`), die `market_chart/range` erwartet.

In [ ]:
# Build major exchange universe
cmc_top_exchanges_df = fetch_top_exchange_names_from_cmc(TOP_EXCHANGES_N)
coingecko_exchange_map_df = fetch_coingecko_exchange_id_map()

top_exchange_map_df = map_cmc_exchanges_to_coingecko_ids(
    cmc_exchanges_df=cmc_top_exchanges_df,
    coingecko_exchange_map_df=coingecko_exchange_map_df,
)

top_exchange_ids = (
    top_exchange_map_df.loc[
        top_exchange_map_df["coingecko_exchange_id"].notna(),
        "coingecko_exchange_id",
    ]
    .astype(str)
    .tolist()
)

print(f"Top exchange names requested: {TOP_EXCHANGES_N}")
print(f"Mapped CoinGecko exchange ids: {len(top_exchange_ids)}")
display(top_exchange_map_df)

In [ ]:
# Date range + output/cache dirs
ensure_output_dirs()

date_list = make_date_list(START_DATE, END_DATE)
from_ts = int(pd.Timestamp(START_DATE, tz="UTC").timestamp())
to_ts = int(pd.Timestamp(END_DATE, tz="UTC").timestamp()) + 86_400  # include the full end date

print("Date range validated.")
print(f"START_DATE: {START_DATE}")
print(f"END_DATE:   {END_DATE}")
print(f"Days in window: {len(date_list)}")
print(f"from_ts={from_ts} to_ts={to_ts}")

## 11 · Ausführung: Pipeline (Schritte a–g)

Eine Zelle pro Schritt aus Abschnitt 1, in Reihenfolge. Jede batchweise Zelle (b, e)
druckt Zwischenstände nach je `BATCH_SIZE` Coins mit — läuft alles gecacht, kostet ein
erneutes Ausführen keine zusätzlichen Credits.

In [ ]:
# Step (a): today's candidate pool
candidate_pool_size = DEV_CANDIDATE_LIMIT or TOP_N_CANDIDATES
candidates_df = fetch_top_n_candidates(candidate_pool_size)

n_before_stablecoin_filter = len(candidates_df)
excluded_stablecoins = candidates_df.loc[candidates_df["coingecko_id"].isin(STABLECOIN_DENYLIST)]
candidates_df = candidates_df.loc[~candidates_df["coingecko_id"].isin(STABLECOIN_DENYLIST)].reset_index(drop=True)

print(f"Candidate pool: {len(candidates_df)} coins (today's top {candidate_pool_size} by market cap, "
      f"{len(excluded_stablecoins)} stablecoins excluded)")
if not excluded_stablecoins.empty:
    print(f"Excluded: {excluded_stablecoins['symbol'].tolist()}")
display(candidates_df.head())

In [ ]:
# Step (b): historical time series per candidate (1 call/coin, full window, batched + cached)
market_charts, market_chart_errors = fetch_market_chart_for_candidates(
    candidate_ids=candidates_df["coingecko_id"].tolist(),
    from_ts=from_ts,
    to_ts=to_ts,
    batch_size=BATCH_SIZE,
)

if market_chart_errors:
    print(f"\n{len(market_chart_errors)} candidates failed to fetch market_chart data:")
    display(pd.DataFrame(market_chart_errors))

In [ ]:
# Step (c): reconstruct the daily panel from cached raw responses, apply filters 1+2
daily_panel_df = build_daily_panel_from_market_charts(market_charts)
eligible_daily_df = apply_daily_filters(
    panel_df=daily_panel_df,
    top_n_daily=TOP_N_DAILY,
    min_volume_24h_usd=MIN_VOLUME_24H_USD,
)

print(f"Daily panel rows (all candidates x all days): {len(daily_panel_df)}")
print(f"Eligible rows after filter 1 (top {TOP_N_DAILY}) + filter 2 (volume > {MIN_VOLUME_24H_USD:,.0f}): {len(eligible_daily_df)}")
display(eligible_daily_df.head())

# Correctness spot-check: BTC price must actually vary day-to-day (disproves the old, confirmed-broken
# behavior where /coins/markets silently ignored the `date` param and froze prices for weeks at a time).
_btc_check = daily_panel_df.loc[daily_panel_df["coingecko_id"] == "bitcoin"].sort_values("date")
if not _btc_check.empty:
    _btc_prices = pd.to_numeric(_btc_check["price_usd"], errors="coerce")
    print(f"\nBTC price sanity check: {_btc_prices.nunique()} distinct values across {len(_btc_prices)} days "
          f"(median day-over-day change: {_btc_prices.pct_change().abs().median():.4%})")

In [ ]:
# Step (d): union of coins eligible on at least one day, with static metadata (reused from step a, no extra calls)
union_ids = sorted(eligible_daily_df["coingecko_id"].unique().tolist())
union_static_df = candidates_df.loc[candidates_df["coingecko_id"].isin(union_ids)].reset_index(drop=True)

print(f"Union of ever-eligible coins (filters 1+2, whole window): {len(union_ids)}")
display(union_static_df.head())

In [ ]:
# Step (e): filter 3, current state only - top-exchange coverage + Binance-symbol resolution
# (top_exchange_ids computed earlier from the CMC/CoinGecko exchange mapping cell)
filter3_df, filter3_errors = fetch_filter3_data_for_union(
    union_ids=union_ids,
    exchange_ids=top_exchange_ids,
    batch_size=BATCH_SIZE,
)

if filter3_errors:
    print(f"\n{len(filter3_errors)} union coins failed the filter-3 ticker fetch:")
    display(pd.DataFrame(filter3_errors))

display(filter3_df.head())

In [ ]:
# Step (f)+(g): build final fixed assets.parquet (with binance_symbol) and asset_daily.parquet
existing_ids_df = load_existing_asset_ids()

asset_df = build_final_assets_df(
    union_static_df=union_static_df,
    filter3_df=filter3_df,
    min_top_exchange_count=MIN_TOP_EXCHANGE_COUNT,
    existing_ids_df=existing_ids_df,
)

asset_daily_df = build_asset_daily_df_final(eligible_daily_df, asset_df)

n_binance_matched = asset_df["binance_symbol"].notna().sum()
n_reused_ids = asset_df["coingecko_id"].isin(existing_ids_df["coingecko_id"]).sum() if not existing_ids_df.empty else 0
print(f"Union size (filters 1+2):            {len(union_ids)}")
print(f"Final assets (after filter 3, >= {MIN_TOP_EXCHANGE_COUNT} exchanges): {len(asset_df)}")
print(f"Of which resolved to a Binance symbol: {n_binance_matched}/{len(asset_df)}")
print(f"Of which reused an asset_id from the previous run: {n_reused_ids}/{len(asset_df)}")

unresolved = asset_df.loc[asset_df["binance_symbol"].isna(), ["coingecko_id", "symbol", "name"]]
if not unresolved.empty:
    print("\nAssets with no Binance USDT/stable pair (excluded from OHLCV fetching downstream):")
    display(unresolved)

## 12 · Validieren, Anzeigen & Speichern

`validate_final_tables()` muss fehlerfrei durchlaufen, bevor irgendetwas auf Platte
geschrieben wird. Die Anzeige zeigt `asset_df` vollständig (überschaubare Grösse), für
`asset_daily_df` aber nur eine Stichprobe + Kennzahlen statt aller ~19'000 Zeilen — die
volle Zeitreihe pro Tag ist stattdessen der Plot in Abschnitt 13.

In [ ]:
# Validation
asset_df = ensure_asset_columns(asset_df)
asset_daily_df = enforce_columns(asset_daily_df, ASSET_DAILY_COLUMNS)

validate_final_tables(asset_df, asset_daily_df)

print("All validation checks passed.")

In [ ]:
# Output tables
market_chart_error_df = pd.DataFrame(market_chart_errors)
filter3_error_df = pd.DataFrame(filter3_errors)

print(f"asset_df: {asset_df.shape}")
display(asset_df)

# asset_daily_df has ~one row per (day, eligible asset) -> too large to dump in full
# (see the plot in section 13 for the full time series). Show a small sample instead.
print(f"\nasset_daily_df: {asset_daily_df.shape} — showing first/last 5 rows")
display(pd.concat([asset_daily_df.head(5), asset_daily_df.tail(5)]))

if not market_chart_error_df.empty:
    print("market_chart fetch errors:")
    display(market_chart_error_df)

if not filter3_error_df.empty:
    print("Filter-3 ticker fetch errors:")
    display(filter3_error_df)

In [ ]:
# Save final tables
ensure_output_dirs()

asset_df.to_parquet(ASSETS_PATH, index=False)
asset_daily_df.to_parquet(ASSET_DAILY_PATH, index=False)

print(f"Saved asset table: {ASSETS_PATH}")
print(f"Saved asset_daily table: {ASSET_DAILY_PATH}")
print(f"Total asset rows stored: {len(asset_df)}")
print(f"Total asset_daily rows stored: {len(asset_daily_df)}")

## 13 · Visualisierung

Anzahl eligible Assets pro Tag (Filter 1+2) über den ganzen Zeitraum — reine Kontrolle,
verändert keine Daten. Ein sinkender Trend hier ist ein Signal, das man verstehen sollte
(z.B. insgesamt weniger Coins über der Volumen-Schwelle in jüngerer Zeit), nicht
zwingend ein Fehler.

In [ ]:
# Plot: how many eligible assets per day
daily_counts = asset_daily_df.groupby("date")["asset_id"].nunique().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.to_datetime(daily_counts.index), daily_counts.values, color="#2a78d6", linewidth=1.5)
ax.fill_between(pd.to_datetime(daily_counts.index), daily_counts.values, color="#2a78d6", alpha=0.1)

ax.set_title("Anzahl eligible Assets pro Tag (Filter 1+2)")
ax.set_xlabel("Datum")
ax.set_ylabel("Anzahl Assets")
ax.grid(True, axis="y", alpha=0.3)
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Min: {daily_counts.min()} | Max: {daily_counts.max()} | Mittelwert: {daily_counts.mean():.1f}")